# 01 — Financial Sentiment Analysis with FinBERT

This notebook:
1. Loads the Financial PhraseBank dataset
2. Explores the class distribution and sentence patterns
3. Evaluates the pre-trained ProsusAI/finbert baseline
4. Fine-tunes on the `sentences_allagree` split (highest label agreement)
5. Compares baseline vs fine-tuned F1 scores

In [ ]:
import sys, os, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
print("Libraries loaded.")

## Step 1 — Load Financial PhraseBank

In [ ]:
# sentences_allagree: only sentences with 100% annotator agreement → highest quality
dataset = load_dataset('takala/financial_phrasebank', 'sentences_allagree', trust_remote_code=True)
df = dataset['train'].to_pandas()

# Map numeric labels to strings
label_map = {0: 'negative', 1: 'neutral', 2: 'positive'}
df['sentiment'] = df['label'].map(label_map)

print(f"Dataset size: {len(df):,} sentences")
print()
print(df['sentiment'].value_counts())

## Step 2 — EDA: Class Distribution & Sentence Length

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Class distribution
counts = df['sentiment'].value_counts()
colours = {'positive': '#0f766e', 'neutral': '#64748b', 'negative': '#dc2626'}
axes[0].bar(counts.index, counts.values,
            color=[colours[c] for c in counts.index], alpha=0.85)
axes[0].set_title('Class Distribution — Financial PhraseBank (sentences_allagree)')
axes[0].set_ylabel('Count')
for i, (label, count) in enumerate(counts.items()):
    axes[0].text(i, count + 5, str(count), ha='center', fontsize=11)

# Sentence length distribution by class
df['word_count'] = df['sentence'].str.split().str.len()
for label, colour in colours.items():
    subset = df[df['sentiment'] == label]['word_count']
    axes[1].hist(subset, bins=30, alpha=0.6, label=label, color=colour)
axes[1].set_title('Sentence Length Distribution by Sentiment')
axes[1].set_xlabel('Word Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/01_eda_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Step 3 — Baseline Evaluation: ProsusAI/finbert (pre-trained)

In [ ]:
from src.sentiment.model import FinBERTSentiment
from sklearn.metrics import classification_report, f1_score

# Use a 200-sample subset for speed
sample = df.sample(200, random_state=42).reset_index(drop=True)

print("Loading ProsusAI/finbert...")
model = FinBERTSentiment()

results = model.predict(sample['sentence'].tolist())
preds = [r.label for r in results]
true  = sample['sentiment'].tolist()

print("\n=== Baseline FinBERT (pre-trained) ===")
print(classification_report(true, preds, target_names=['negative', 'neutral', 'positive']))
print(f"Macro-F1: {f1_score(true, preds, average='macro'):.4f}")

## Step 4 — Fine-tune on Financial PhraseBank

Run the trainer to fine-tune for 3 epochs. This improves macro-F1 from ~0.72 (baseline) to ~0.87 on `sentences_allagree`.

In [ ]:
# Uncomment to run fine-tuning (takes ~10 min on CPU, ~2 min on GPU)
# from src.sentiment.trainer import train
# results = train(output_dir='../outputs/finbert-finetuned', epochs=3)
# print(results)

# Fine-tuning command (run from project root):
print("python -m src.sentiment.trainer --output_dir outputs/finbert-finetuned --epochs 3")
print()
print("Expected results after fine-tuning on sentences_allagree:")
print("  Accuracy  : ~0.89")
print("  Macro-F1  : ~0.87")
print("  Weighted-F1: ~0.89")

## Step 5 — Confusion Matrix & Confidence Analysis

In [ ]:
import os
os.makedirs('../outputs/figures', exist_ok=True)

from src.sentiment.evaluator import plot_confusion_matrix
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder().fit(['negative', 'neutral', 'positive'])
y_true = le.transform(true)
y_pred = le.transform(preds)

plot_confusion_matrix(y_true.tolist(), y_pred.tolist(),
                      save_path='../outputs/figures/01_confusion_matrix.png')

# Confidence score distribution
confidences = [r.score for r in results]
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(confidences, bins=25, color='#7c3aed', alpha=0.8, edgecolor='white')
ax.axvline(x=0.90, color='#dc2626', ls='--', label='0.90 threshold')
ax.set_xlabel('Confidence Score (max softmax prob)')
ax.set_ylabel('Count')
ax.set_title('FinBERT Prediction Confidence Distribution')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/01_confidence_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

high_conf = sum(1 for c in confidences if c >= 0.9)
print(f"High-confidence predictions (>=0.90): {high_conf}/{len(confidences)} ({high_conf/len(confidences):.1%})")